In [ ]:
import pandas as pd
import numpy as np

from math import ceil

import sys, os, time, random, re, csv, json, argparse, torch #,copy , indexer, evaluate

from datetime import datetime
from datasets import Dataset, Value, concatenate_datasets 
from sklearn.metrics import mean_squared_error, f1_score, accuracy_score, precision_score, recall_score, classification_report
from scipy.special import expit

from torch.nn import functional as F#, BCEWithLogitsLoss
from torch.utils.data import WeightedRandomSampler

from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, 
                          BertForSequenceClassification, BertModel, EarlyStoppingCallback, AdamW,
                          PreTrainedModel, Trainer, TrainingArguments, get_scheduler)

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object #, text_cleansing 

In [ ]:
value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')
value_set

In [ ]:
value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)
value_test_set

In [ ]:
test_set = value_set.iloc[value_test_set]
test_set

In [ ]:
test_set = test_set[test_set['label']==1]
test_set

In [ ]:
test_set[test_set.duplicated('scenario')]

In [ ]:
test_set[test_set['uid']=='A19080']

In [ ]:
test_set.groupby(by=["value"]).sum('label')

In [ ]:
test_subset = test_set.groupby('value').head(99)

In [ ]:
test_subset

In [ ]:
test_subset.groupby('value').sum('label')

In [ ]:
test_subset[test_subset.duplicated('scenario',keep=False)].sort_values('scenario')

In [ ]:
#valuenet['label'] = abs(valuenet['label'])
test_subset_pivot = test_subset.pivot(index='scenario', columns='value', values='label').fillna(0).astype(int)
test_subset_pivot.columns.name = None
test_subset_pivot.reset_index(inplace=True)
test_subset_pivot

In [ ]:
test_subset_pivot.columns

In [ ]:
test_subset_f = pd.melt(test_subset_pivot, id_vars=['scenario'],
                   value_vars=['ACHIEVEMENT', 'BENEVOLENCE', 'CONFORMITY', 'HEDONISM','POWER',
                               'SECURITY', 'SELF-DIRECTION', 'STIMULATION', 'TRADITION','UNIVERSALISM'],
                   ignore_index=True)
test_subset_f.reset_index(inplace=True)
test_subset_f.rename(columns={"index": "uid","variable":"value","value":"label"}, inplace=True)
test_subset_f

In [ ]:
test_subset_f.groupby('value').count()#.sum('label')

In [ ]:
test_subset_f.to_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test_strat.csv",
                     header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")